# Simulation — controlled recovery against known truth

Kept separate from the atlas notebook on purpose. Simulation has ground truth, which the atlas does
not, and it has **twice mispredicted the atlas**: an initialisation that helped at c=2 in simulation
made the real atlas worse (effective K 36% -> 29%), and the same happened for waypoints+phi. Treat
conclusions here as advisory for the atlas, never as a substitute.

Shared measurement code lives in `figures/figure_common.py`.

In [ ]:
import os, sys, json, glob, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt

FIGDIR = "/gpfs/commons/home/kisaev/Leaflet-analysis/GenomeBiology_Submission2026/figures"
sys.path.insert(0, FIGDIR)
import figure_common as fc
try:
    import leaflet_theme as th; th.apply_theme()
except Exception as e:
    print("theme unavailable:", e)

SIM = fc.SIM_DIR
BENCH = f"{FIGDIR}/FIGURE2_benchmark"
print("simulated datasets:", len([f for f in os.listdir(SIM) if f.endswith('.h5ad')]))

## 1. The benchmark sweep

`benchmark_effect_sweep.csv` — 4 effect levels x 3 seeds, all six method arms. The x-axis is the
**median ΔΨ over differential cassettes only**: ΔΨ = max_k Ψ_k − min_k Ψ_k, a RANGE across programs,
not a pairwise average.

In [ ]:
bench = pd.read_csv(f"{BENCH}/benchmark_effect_sweep.csv")
print(bench.shape, "|", sorted(bench.model.unique()), "|", sorted(bench.likelihood.unique()))
bench.groupby(["effect","model","likelihood"])[["psi_r","sil","auroc","spearman_delta"]].mean().round(3)

### Effect-size recovery — the headline

LeafletFA rises with ΔΨ; **both NMF variants sit at ~0 at every effect size**, i.e. they recover no
quantitative effect-size information at all.

In [ ]:
METRIC = "spearman_delta"        # try: sil, auroc, psi_r
fig, ax = plt.subplots(figsize=(5.2, 3.4))
for (mdl, lik), g in bench.groupby(["model","likelihood"]):
    if mdl == "LeafletFA" and lik not in ("bb",):   # reference config only
        continue
    m = g.groupby("effect")[METRIC].agg(["mean","sem"])
    ax.errorbar(m.index, m["mean"], yerr=m["sem"], marker="o", capsize=3,
                label=f"{mdl} ({lik})")
ax.set_xlabel("simulated effect size"); ax.set_ylabel(METRIC); ax.legend(frameon=False, fontsize=8)
plt.show()

### Careful with Ψ recovery

Read it with the other two. At the smallest effect size NMF mean-impute scores ~0.92 on Ψ recovery
while having ~0 effect-size recovery and negative silhouette — predicting each junction's mean
correlates with every program profile while capturing no program structure.

In [ ]:
bench[bench.effect == bench.effect.min()].groupby(["model","likelihood"])[
    ["psi_r","sil","spearman_delta"]].mean().round(3)

## 2. Overdispersion — why reproducibility has a ceiling

Same design at c = ∞ / 20 / 2. This is the arm that justifies the Beta-Binomial, and the one that
explains the atlas's seed behaviour.

In [ ]:
od = [f for f in glob.glob(f"{BENCH}/rerun_logs/odbench_*.csv") if "detinit" not in f]
if od:
    O = pd.concat([pd.read_csv(f) for f in od])
    display(O[O.model=="LeafletFA"].groupby(["sim_bb_conc","likelihood"])[
        ["psi_r","sil","auroc","spearman_delta"]].mean().round(3))
else:
    print("overdispersion shards not found")

## 3. Seed consistency, with truth available

`A13` measured: c = ∞/20/2 gives 100% / 83% / 42% reproducible φ directions, and accuracy against the
KNOWN φ degrades in step (0.990 → 0.959 → 0.788). Seeds do not merely disagree — at high dispersion
they are all further from the truth.

In [ ]:
p = f"{SIM}/A13_overdispersion.csv"
display(pd.read_csv(p).round(3)) if os.path.exists(p) else print("A13 csv not found")

## 4. Cell-cell similarity across seeds — permutation-invariant

Needs no program matching. `A26` writes the matrices; ordering is by TRUE program, so a reproducible
fit shows the same blocks in every seed and in the truth panel.

In [ ]:
p = f"{SIM}/A26_cellcell.npz"
if not os.path.exists(p):
    print("A26 not finished yet")
else:
    z = np.load(p, allow_pickle=True)
    LEVEL = "bb2"                       # binom | bb20 | bb2
    order = np.argsort(np.asarray(z[f"order_{LEVEL}"]))[:400]
    keys = [f"S_{LEVEL}_seed0", f"S_{LEVEL}_seed1", f"S_{LEVEL}_true"]
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.9))
    for a, k in zip(ax, keys):
        im = a.imshow(np.asarray(z[k])[np.ix_(order, order)], cmap="BrBG", vmin=-1, vmax=1)
        a.set_title(k.replace(f"S_{LEVEL}_","")); a.set_xticks([]); a.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.02, label="cell-cell similarity")
    plt.show()
    print(pd.DataFrame(z["summary"]).round(3).to_string(index=False))

## Co-clustering consistency (consensus clustering)

Cluster φ separately under each seed, then ask **how often a given pair of cells lands in the same
cluster**. This is the standard consensus-clustering construction (Monti et al., *Machine Learning*
2003): the **co-association matrix** `C[i,j] = fraction of runs placing i and j together`.

It is the right tool here for two reasons. It is **permutation-invariant** — cluster labels are
arbitrary in exactly the way program labels are, and asking about *pairs* removes the need to match
anything. And it degrades gracefully: a partition can shift its boundaries a little without the pair
structure collapsing, which a raw ARI would punish.

Three readouts:

| | meaning |
|---|---|
| **co-association distribution** | should be **bimodal** — pairs are either reliably together (≈1) or reliably apart (≈0). Mass in the middle is instability |
| **PAC** (Proportion of Ambiguous Clustering) | fraction of pairs with co-association strictly between 0.1 and 0.9 (Șenbabaoğlu et al., *Sci Rep* 2014). **Lower is better**; it is the standard scalar summary of that bimodality |
| **pairwise ARI** | agreement between the raw partitions. Reported alongside because it is what most papers quote, and because it is *harsher* than the pair view |

With only 3 seeds the co-association values can only be 0, 1/3, 2/3 or 1, so the histogram is coarse —
PAC counts the 1/3 and 2/3 pairs. More seeds would give a finer curve.

In [ ]:
# Refit the simulated data under several seeds (or reuse fits you already have in memory).
import anndata as ad
sys.path.insert(0, "/gpfs/commons/home/kisaev/LeafletFA")
from leafletfa.minibatch import LeafletFAMinibatch

LEVEL = "bb2"                      # binom | bb20 | bb2
N_SEEDS = 3
a = ad.read_h5ad(f"{SIM}/sim_heart_K5_{LEVEL}_eff0.25_frac0.3_seed0.h5ad")
g = lambda k: np.asarray(a.layers[k].todense() if hasattr(a.layers[k], "todense")
                         else a.layers[k], np.float32)
junc, clus = g("cell_by_junction_matrix"), g("cell_by_cluster_matrix")
TRUE_PHI = np.asarray(a.obsm["true_phi"], float)
Ktrue = TRUE_PHI.shape[1]

_PHI = {}
for s in range(N_SEEDS):
    m = LeafletFAMinibatch(junc, clus, K=Ktrue, junc_specific_prior=True, likelihood="betabinomial")
    m.fit(num_steps=5000, subsample_size=2048, lr=0.05, gamma=0.1, seed=s, verbose=False)
    _PHI[s] = m.phi_mean().cpu().numpy()
_TASKS = sorted(_PHI); _PHI_N = TRUE_PHI.shape[0]
N_CLUST = Ktrue
print(f"simulation {LEVEL}: {N_SEEDS} seeds, {_PHI_N:,} cells, K={Ktrue}")

In [ ]:
# ── consensus / co-clustering consistency ─────────────────────────────────────────
# N^2 pairs, so subsample. The E-step is per-cell independent, so this does not bias anything.
N_SUB   = 3000
METHOD  = "kmeans"        # "kmeans" (fast) or "leiden" (needs scanpy)
N_CLUST = 15              # kmeans only; leiden uses RESOLUTION
RESOLUTION = 1.0

rng = np.random.default_rng(0)
sub = np.sort(rng.choice(_PHI_N, size=min(N_SUB, _PHI_N), replace=False))

def cluster_phi(phi, method=METHOD, seed=0):
    X = np.nan_to_num(np.asarray(phi, np.float32))
    if method == "kmeans":
        from sklearn.cluster import KMeans
        return KMeans(n_clusters=N_CLUST, n_init=10, random_state=seed).fit_predict(X)
    import scanpy as sc, anndata as ad
    a = ad.AnnData(X)
    sc.pp.neighbors(a, n_neighbors=15, use_rep="X", random_state=seed)
    sc.tl.leiden(a, resolution=RESOLUTION, random_state=seed, key_added="l",
                 flavor="igraph", n_iterations=2, directed=False)
    return a.obs["l"].astype(str).values

parts = {t: cluster_phi(_PHI[t][sub], seed=0) for t in _TASKS}   # SAME clustering seed:
# only the model seed varies, so any disagreement comes from phi, not from the clusterer.
print({t: len(set(p)) for t, p in parts.items()}, "clusters per seed")

# co-association: fraction of runs putting each pair together
n = len(sub)
C = np.zeros((n, n), dtype=np.float32)
for p in parts.values():
    lab = pd.factorize(p)[0]
    C += (lab[:, None] == lab[None, :]).astype(np.float32)
C /= len(parts)
iu = np.triu_indices(n, 1)
co = C[iu]

PAC = float(np.mean((co > 0.1) & (co < 0.9)))
from sklearn.metrics import adjusted_rand_score as ari
aris = [ari(parts[a], parts[b]) for a, b in fc.pairs(_TASKS)]

print(f"\nPAC (ambiguous pairs, lower is better) : {PAC:.4f}")
print(f"pairs always together                  : {np.mean(co == 1):.4f}")
print(f"pairs never together                   : {np.mean(co == 0):.4f}")
print(f"pairwise ARI between partitions        : {np.mean(aris):.4f}  {np.round(aris,3).tolist()}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(co, bins=np.linspace(0, 1, len(parts) * 2 + 3), color="#006865")
ax[0].set_xlabel("co-association (fraction of seeds together)")
ax[0].set_ylabel("cell pairs"); ax[0].set_yscale("log")
ax[0].set_title(f"bimodal = stable   |   PAC = {PAC:.3f}")
o = np.argsort(pd.factorize(parts[_TASKS[0]])[0])
im = ax[1].imshow(C[np.ix_(o, o)], cmap="BrBG", vmin=0, vmax=1, interpolation="nearest")
ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].set_title("co-association matrix")
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

In [ ]:
# Simulation only: does the consensus recover the TRUE grouping, or just agree with itself?
from sklearn.metrics import adjusted_rand_score as ari
true_lab = TRUE_PHI.argmax(1)[sub]
vs_truth = [ari(parts[t], true_lab) for t in _TASKS]
print(f"ARI vs TRUE program:  {np.mean(vs_truth):.4f}  {np.round(vs_truth,3).tolist()}")
print(f"ARI between seeds  :  {np.mean(aris):.4f}")
print("\nHigh between-seed ARI with low ARI-vs-truth would mean the runs agree on a WRONG grouping.")

## 5. Fit your own

Refit a simulated dataset with whatever settings you want and score against truth.

In [ ]:
# sys.path.insert(0, "/gpfs/commons/home/kisaev/LeafletFA")
# import anndata as ad
# from leafletfa.minibatch import LeafletFAMinibatch
# a = ad.read_h5ad(f"{SIM}/sim_heart_K5_bb2_eff0.25_frac0.3_seed0.h5ad")
# g = lambda k: np.asarray(a.layers[k].todense() if hasattr(a.layers[k],'todense') else a.layers[k], np.float32)
# m = LeafletFAMinibatch(g("cell_by_junction_matrix"), g("cell_by_cluster_matrix"),
#                        K=5, junc_specific_prior=True, likelihood="betabinomial")
# m.fit(num_steps=5000, subsample_size=2048, lr=0.05, gamma=0.1, seed=0)
# phi_true = np.asarray(a.obsm["true_phi"], float)
# print("phi subspace agreement with truth:", fc.cca(m.phi_mean().cpu().numpy(), phi_true).mean())

## Scratch

In [ ]:
#